In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Doan2108/contract-risk-qwen2.5-3b-merged"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Loaded")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/6.17G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded


In [2]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class AnalyzeRequest(BaseModel):
    content: str

In [3]:
import os
import json
import re
import logging
from typing import List, Optional

import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM

# ==========================================================
# Logging
# ==========================================================

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("ai_service")

# ==========================================================
# FastAPI
# ==========================================================

app = FastAPI(
    title="RiskDL AI Inference Service",
    version="1.0.0"
)

# ==========================================================
# Model Loading
# ==========================================================

MODEL_NAME = os.environ.get(
    "MODEL_NAME",
    "Doan2108/contract-risk-qwen2.5-3b-merged"
)

model = None
tokenizer = None

try:

    logger.info(f"Loading tokenizer: {MODEL_NAME}")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )

    logger.info(f"Loading model: {MODEL_NAME}")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    model.eval()

    logger.info("Model loaded successfully!")

except Exception as e:

    logger.exception(
        f"Failed to load model: {e}"
    )

# ==========================================================
# Request Models
# ==========================================================

class ClauseInput(BaseModel):
    title: str
    content: str


class ExtractedEntityInput(BaseModel):
    clause_title: str
    entity_type: str
    entity_value: str
    normalized_value: str = ""
    confidence_score: float = 1.0


class RiskRuleInput(BaseModel):
    name: str
    description: Optional[str] = ""


class AnalyzeRequest(BaseModel):
    clauses: List[ClauseInput]
    extracted_entities: List[ExtractedEntityInput] = []
    risk_rules: List[RiskRuleInput] = []

# ==========================================================
# Response Models
# ==========================================================

class FindingOutput(BaseModel):
    clause_title: str
    risk_category: str
    risk_level: str
    explanation: str
    recommendation: str
    disadvantaged_party: Optional[str] = None


class EntityOutput(BaseModel):
    clause_title: str
    entity_type: str
    entity_value: str
    normalized_value: str = ""
    confidence_score: float = 1.0


class AnalyzeResponse(BaseModel):
    overall_score: int
    summary: str
    findings: List[FindingOutput]
    entities: List[EntityOutput] = []

# ==========================================================
# Helpers
# ==========================================================

def clean_and_parse_json(text: str):

    text = text.strip()

    match = re.search(
        r"```json\s*(.*?)\s*```",
        text,
        re.DOTALL
    )

    if match:
        text = match.group(1)

    else:
        start = text.find("{")
        end = text.rfind("}")

        if start != -1 and end != -1:
            text = text[start:end + 1]

    return json.loads(text)


def generate_response(messages: list) -> str:

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

# ==========================================================
# AI Analysis
# ==========================================================

def run_ai_analysis(
    clauses: List[ClauseInput],
    extracted_entities: List[ExtractedEntityInput],
    risk_rules: List[RiskRuleInput] = []
) -> AnalyzeResponse:

    findings = []

    total_score = 0
    scores_count = 0

    rules_instruction = ""

    if risk_rules:

        rules_instruction += (
            "\n\nDanh sách các loại rủi ro hiện có:\n"
        )

        for r in risk_rules:

            rules_instruction += (
                f"- '{r.name}': {r.description}\n"
            )

    for clause in clauses:

        prompt_content = f"""
TIÊU ĐỀ:
{clause.title}

NỘI DUNG:
{clause.content}
"""

        messages = [
            {
                "role": "system",
                "content": "Bạn là chuyên gia phân tích rủi ro hợp đồng pháp lý tại Việt Nam. "
                "Hãy đóng vai trò là một Luật sư cực kỳ nghiêm khắc, kỹ tính và luôn bảo vệ quyền lợi của Bên thuê/Bên mua. "
                "Nhiệm vụ của bạn là đọc kỹ điều khoản hợp đồng và phát hiện tất cả các lỗi, điểm bất lợi, rủi ro tiềm ẩn hoặc sự bất đối xứng quyền lợi. "
                "Luôn trả về JSON thuần túy với các trường sau: "
                "\"risk_category\" (str: Ví dụ 'Limitation of Liability Risk', 'Payment Risk', 'Unbalanced Termination Clause', hoặc tên rủi ro phù hợp), "
                "\"severity\" (str: 'NONE', 'LOW', 'MEDIUM', 'HIGH', 'CRITICAL'), "
                '"risk_score" (int 0-100), '
                '"explanation" (str: Giải thích chi tiết bằng tiếng Việt lý do điều khoản này có rủi ro hoặc bất lợi), '
                '"recommendation" (str: Đề xuất sửa đổi cụ thể bằng tiếng Việt để giảm thiểu rủi ro), '
                "\"disadvantaged_party\" (str: Bên gặp bất lợi, ví dụ 'Bên B', hoặc null). "
                f'Hãy suy luận cực kỳ chặt chẽ để tìm ra rủi ro. Nếu điều khoản thực sự hoàn toàn an toàn và không có bất kỳ rủi ro nào, hãy đặt "severity": "NONE", "risk_score": 0, "risk_category": "Safe" và "disadvantaged_party": null.{rules_instruction}',
            },
            {"role": "user", "content": prompt_content},
        ]

        response = generate_response(messages)

        logger.info(
            f"Raw response for '{clause.title}': {response}"
        )

        try:

            parsed = clean_and_parse_json(
                response
            )

            risk_cat = parsed.get(
                "risk_category",
                "Unknown Risk"
            )

            severity = parsed.get(
                "severity",
                "NONE"
            )

            risk_score = int(
                parsed.get(
                    "risk_score",
                    0
                )
            )

            explanation = parsed.get(
                "explanation",
                ""
            )

            recommendation = parsed.get(
                "recommendation",
                ""
            )

            disadvantaged = parsed.get(
                "disadvantaged_party"
            )

            if severity != "NONE":

                findings.append(
                    FindingOutput(
                        clause_title=clause.title,
                        risk_category=risk_cat,
                        risk_level=severity,
                        explanation=explanation,
                        recommendation=recommendation,
                        disadvantaged_party=disadvantaged
                    )
                )

                total_score += risk_score
                scores_count += 1

        except Exception as e:

            logger.error(
                f"Parse error: {e}"
            )

    overall_score = (
        int(total_score / scores_count)
        if scores_count > 0
        else 0
    )

    summary = (
        f"Scanned {len(clauses)} clauses. "
        f"Found {len(findings)} risks."
    )

    return AnalyzeResponse(
        overall_score=overall_score,
        summary=summary,
        findings=findings
    )

# ==========================================================
# API
# ==========================================================

@app.post(
    "/api/v1/analyze",
    response_model=AnalyzeResponse
)
async def analyze_contract(
    payload: AnalyzeRequest
):

    if not payload.clauses:

        raise HTTPException(
            status_code=400,
            detail="No clauses provided."
        )

    if model is None:

        raise HTTPException(
            status_code=503,
            detail="Model not loaded."
        )

    return run_ai_analysis(
        payload.clauses,
        payload.extracted_entities,
        payload.risk_rules
    )


@app.get("/health")
async def health_check():

    return {
        "status":
        "healthy"
        if model is not None
        else "unhealthy",

        "model_loaded":
        model is not None,

        "model_name":
        MODEL_NAME
    }


SyntaxError: invalid syntax (4229678847.py, line 1)

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest_asyncio

In [ ]:
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run, daemon=True).start()

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("TOKEN_NGROK")

public_url = ngrok.connect(8000)

print(public_url)

In [ ]:
requests.post(
    "https://xxxxx.ngrok-free.app/api/v1/analyze",
    json=payload,
    timeout=300
)